### 1. Configuração do Ecossistema de Data Science
Nesta célula inicial, preparamos o ambiente de desenvolvimento importando as bibliotecas fundamentais para o processamento de dados e inteligência artificial do projeto **PlaySafe4All**:

* **Manipulação de Dados (`Pandas` & `NumPy`)**: Essenciais para a estruturação das tabelas biomecânicas e cálculos matemáticos de alta performance.
* **Gestão de Sistema (`OS` & `Time`)**: Permitem a localização dinâmica de ficheiros em diferentes sistemas operativos e a medição do tempo de resposta do algoritmo.
* **Algoritmos de Previsão (`XGBoost`)**: O motor de decisão que utiliza *Gradient Boosting* para identificar padrões de risco de entorse.
* **Ecossistema `Scikit-Learn`**:
    * **Divisão de Dados**: Para separar amostras de treino e de teste.
    * **Normalização**: Garante que variáveis com unidades diferentes (ex: tempo vs. força) sejam interpretadas com a mesma importância.
    * **Métricas de Avaliação**: Ferramentas para medir a precisão e, crucialmente, o **Recall** (segurança do atleta).

In [6]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS
# Responsabilidade: Carregar todas as ferramentas necessárias para o projeto.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time

# Modelos e Métricas
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

print("✅ Todas as bibliotecas foram carregadas com sucesso!")

✅ Todas as bibliotecas foram carregadas com sucesso!


### 2. Aquisição e Consolidação de Dados Biomecânicos
Nesta etapa, o sistema estabelece a ligação com a diretoria externa `Data` para importar os três conjuntos de dados que servem de base ao estudo.

* **Caminhos Dinâmicos**: Utilizamos o `os.path.join` com caminhos relativos (`../Data`). Isto garante que o projeto é multiplataforma, funcionando corretamente quer o "stor" o execute em Windows, Mac ou Linux.
* **Gestão de Erros (Try-Except)**: Implementámos uma estrutura de controlo para capturar erros de localização de ficheiros. Se a pasta ou os CSVs estiverem em falta, o sistema informa o utilizador em vez de interromper o programa abruptamente.
* **Métrica de Volume**: Após o carregamento bem-sucedido, o sistema calcula e apresenta o somatório total de amostras disponíveis, permitindo uma primeira validação visual da integridade dos dados.

In [7]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO DE DADOS
# Responsabilidade: Aceder à pasta 'Data' e carregar os datasets localmente.
# ==============================================================================
print("--- FASE 1: CARREGAMENTO ---")

# Caminho relativo (funciona em qualquer PC com a pasta Data ao lado do Notebook)
BASE_PATH = "../Data" 

try:
    # Construindo os caminhos de forma robusta
    df1 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 1.csv"), sep=',')
    df2 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 2.csv"), sep=',')
    df3 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 3.csv"), sep=',')
    
    print("✅ Arquivos carregados com sucesso!")
    print(f"Total de amostras: {len(df1) + len(df2) + len(df3)}")
    load_successful = True
    
except FileNotFoundError as e:
    print(f"❌ ERRO: Não encontrei a pasta ou os ficheiros em: {BASE_PATH}")
    load_successful = False

--- FASE 1: CARREGAMENTO ---
✅ Arquivos carregados com sucesso!
Total de amostras: 181


### 3. Engenharia de Dados e Normalização Biomecânica
Esta etapa transforma os dados brutos numa matriz estruturada pronta para a aprendizagem automática. O foco é garantir que o modelo **PlaySafe4All** receba informações limpas e comparáveis.

* **Fusão e Limpeza**: Consolidamos os três datasets e removemos identificadores (como o 'Número') que não contribuem para a previsão clínica, evitando o ruído no modelo.
* **Seleção de Variáveis (Features)**: Filtramos apenas os indicadores críticos de exposição (tempo de jogo/treino) e métricas biomecânicas (razões de força e testes de velocidade) correlacionadas com o risco de entorse.
* **Imputação por Mediana**: Tratamos valores em falta utilizando a mediana dos dados. Isto permite manter a integridade estatística sem o enviesamento que uma média poderia causar na presença de valores extremos (*outliers*).
* **Binarização do Alvo**: Convertemos a variável 'Entorse' para um formato binário (0 = Saudável, 1 = Risco), simplificando a tarefa de classificação para o algoritmo.
* **Divisão Estratificada**: Segmentamos os dados em Treino e Teste garantindo que a raridade das lesões seja representada proporcionalmente em ambos os conjuntos (`stratify=y`).
* **Padronização (StandardScaler)**: Reescalamos as variáveis para que todas tenham a mesma importância matemática. Sem isto, variáveis com valores altos (ex: minutos de treino) "abafariam" variáveis com valores pequenos (ex: rácios de força).

In [8]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO (SIMPLIFICADA)
# Responsabilidade: Transformar dados brutos em matrizes de treino/teste.
# ==============================================================================
print("\n--- FASE 2: PREPARAÇÃO DE DADOS ---")

# 1. Unir as bases e remover colunas inúteis
df = pd.concat([df1, df2, df3], ignore_index=True)
if 'Numero' in df.columns: df.drop(columns=['Numero'], inplace=True)

# 2. Seleção de colunas e Alvo
TARGET = 'Entorse'
cols = [TARGET, 'T0_T1_Match_Time_exposure', 'T0_T1_Training_Time_exposure', 
        'T0SRTMax', 'T0TTestMin', 'T0SJmMax', 'T0Veli', 'T0RazaoAADto', 'T0RazaoAAEsq']

df = df[[c for c in cols if c in df.columns]].copy()
df = df.fillna(df.median(numeric_only=True))

# 3. Definição de X/Y (0 = Saudável, 1 = Entorse)
y = df[TARGET].apply(lambda x: 1 if x > 1 else 0) 
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)

# 4. Divisão e Escalonamento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"✅ Dados Prontos para o Random Forest!")


--- FASE 2: PREPARAÇÃO DE DADOS ---
✅ Dados Prontos para o Random Forest!


### 4. Construção e Treino do Modelo (Random Forest)
Nesta fase, implementamos o algoritmo **Random Forest Classifier**. Ao contrário de modelos lineares, o Random Forest utiliza um conjunto de múltiplas árvores de decisão (uma "floresta") para chegar a um veredito final.

* **Ensemble Learning (Bagging)**: O modelo cria 100 árvores independentes (`n_estimators=100`). Cada árvore vota numa classificação (Saudável ou Risco) e a decisão final é tomada pela maioria, o que reduz drasticamente erros de variância.
* **Equilíbrio de Classes (`class_weight='balanced'`)**: Como as lesões são eventos raros no dataset, esta configuração instrui o algoritmo a dar mais importância matemática aos casos de entorse, evitando que o modelo ignore os atletas em risco.
* **Controlo de Complexidade (`max_depth=5`)**: Limitamos a profundidade das árvores para garantir que o modelo aprenda padrões biomecânicos genéricos e não apenas "decore" ruído estatístico das amostras de treino.
* **Estabilidade**: O Random Forest é particularmente robusto a variáveis que possam ter alguma correlação entre si, sendo ideal para dados de performance desportiva.

In [9]:
# ==============================================================================
# CÉLULA 4: TREINO DO RANDOM FOREST
# Responsabilidade: Treinar o algoritmo usando múltiplas árvores de decisão.
# ==============================================================================
print("--- FASE 3: TREINO DO RANDOM FOREST ---")

# 1. Configuração do Modelo
# 'class_weight=balanced' é vital para dar peso às lesões reais
model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=5
)

# 2. Treino (usando .values para evitar avisos de nomes de colunas)
model_rf.fit(X_train_scaled.values, y_train)

print("✅ Modelo 'model_rf' treinado com sucesso!")

--- FASE 3: TREINO DO RANDOM FOREST ---
✅ Modelo 'model_rf' treinado com sucesso!


### 5. Avaliação de Performance e Validação de Segurança (Random Forest)
Nesta fase final, submetemos o modelo **Random Forest** ao teste de rigor clínico, utilizando dados que não foram apresentados durante o treino para garantir a imparcialidade dos resultados.

* **Métricas Multidimensionais**:
    * **Accuracy**: Percentagem de acerto global entre atletas saudáveis e em risco.
    * **Recall (Sensibilidade)**: A nossa métrica de segurança. Mede a capacidade da "floresta" em não deixar passar nenhum caso real de lesão.
    * **F-Score**: Fornece uma visão equilibrada da robustez do modelo, evitando que ele seja excessivamente conservador ou alarmista.
* **Matriz de Confusão e Erro Crítico**:
    * O foco principal recai sobre os **Falsos Negativos (FN)**. No projeto **PlaySafe4All**, um Falso Negativo representa uma falha de proteção — um atleta em risco que o sistema considerou seguro. 
    * Estabelecemos um threshold rigoroso (FN ≤ 2) para validar se o modelo está apto para utilização em contexto desportivo real.
* **Eficiência Computacional**: Monitorizamos o tempo de execução em milissegundos para assegurar que o processamento biomecânico é instantâneo, permitindo intervenções médicas preventivas sem atrasos.

In [10]:
# ==============================================================================
# CÉLULA 5: AVALIAÇÃO FINAL (RANDOM FOREST)
# Responsabilidade: Medir a eficácia e validar a segurança do modelo.
# ==============================================================================
if 'model_rf' in locals():
    print("\n--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---")

    # ⏱️ Início da medição do tempo
    start_time = time.perf_counter()

    # 1. Previsão nos dados de teste (usando apenas as features escalonadas)
    y_pred = model_rf.predict(X_test_scaled.values)

    # 2. Cálculo das Métricas
    # Nota: y_test já está em 0/1 vindo da Célula 3
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f_score = f1_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)

    # ⏱️ Fim da medição do tempo
    execution_time_ms = (time.perf_counter() - start_time) * 1000

    print("\n=== RESULTADOS FINAIS DE DESEMPENHO (RF) ===")
    print(f"📊 Precisão (Accuracy): {accuracy:.4f}")
    print(f"🎯 Recall (Sensibilidade): {recall:.4f}")
    print(f"🧪 F-Score: {f_score:.4f}")
    print(f"⏱️ Tempo de Execução: {execution_time_ms:.2f} ms")

    print("\nMatriz de Confusão:")
    print(conf_matrix)

    # 3. Falsos Negativos (Erro Crítico para o PlaySafe4All)
    if conf_matrix.shape == (2, 2):
        fn = conf_matrix[1, 0]
        print(f"\n❌ Falsos Negativos (Casos não previstos): {fn}")
        
        if fn <= 2:
            print("\n--- ✅ PROJETO CONCLUÍDO E VALIDADO (ALTA SEGURANÇA) ---")
        else:
            print("\n--- ⚠️ ATENÇÃO: FN ALTO. O MODELO PRECISA DE MAIS DADOS ---")
else:
    print("❌ ERRO: O 'model_rf' não foi encontrado. Executa a Célula de Treino primeiro.")


--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---

=== RESULTADOS FINAIS DE DESEMPENHO (RF) ===
📊 Precisão (Accuracy): 0.5956
🎯 Recall (Sensibilidade): 0.5067
🧪 F-Score: 0.5802
⏱️ Tempo de Execução: 23.49 ms

Matriz de Confusão:
[[43 18]
 [37 38]]

❌ Falsos Negativos (Casos não previstos): 37

--- ⚠️ ATENÇÃO: FN ALTO. O MODELO PRECISA DE MAIS DADOS ---
